In [ ]:
pip install mne numpy scipy torch torchvision scikit-learn matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 71.6 MB/s eta 0:00:00


In [ ]:
# Step 1: Unmount if already mounted
!fusermount -u /content/drive

# Step 2: Delete the existing folder
!rm -rf /content/drive

# Step 3: Mount again
from google.colab import drive
drive.mount('/content/drive')

fusermount: failed to unmount /content/drive: No such file or directory
Mounted at /content/drive


In [ ]:
"""
SWCNet v2 — Fixed Implementation, Subject 1 Only Test
======================================================
Fixes from v1:
  1. Bandpass filter 4-40 Hz applied BEFORE windowing
  2. Sliding window ONLY on train data
     Test/Val use CENTER window only (window index 3 of 7)
  3. Normalization: per-channel z-score, fit on TRAIN windows,
     applied to test/val windows
  4. Early stopping patience = 30 (was 10)
  5. LR scheduler: ReduceLROnPlateau (halves LR if val loss plateaus)
  6. Correct normalization order: filter → window → normalize

Paper target for Subject 1 Setup I : ~98.64%
Paper target for Subject 1 Setup II: ~78.16%
"""

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os
import numpy as np
import scipy.io as sio
import scipy.signal as sig_proc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (confusion_matrix, classification_report,
                              f1_score, accuracy_score)
from sklearn.model_selection import KFold

import mne
mne.set_log_level('WARNING')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG  ← set DATA_DIR to your folder
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR    = "/content/drive/MyDrive/BCICIV_2a_gdf"
RESULTS_DIR = "/content/drive/MyDrive/swcnet_results_v2"
os.makedirs(RESULTS_DIR, exist_ok=True)

N_CH        = 22
N_CLS       = 4
FS          = 250

# Trial extraction: cue at t=2s; extract [cue-0.25s, cue+4.25s] = 4.5s
PRE_S       = 0.25
POST_S      = 4.25
TRIAL_S     = PRE_S + POST_S        # 4.5s
TRIAL_SAMP  = int(TRIAL_S * FS)     # 1125 samples

# Sliding window
WIN_S       = 3.0
STEP_S      = 0.25
N_WIN       = 7
WIN_SAMP    = int(WIN_S  * FS)      # 750
STEP_SAMP   = int(STEP_S * FS)      # 62
CENTER_WIN  = 3                     # center window index (0-based)

# Bandpass
FILT_LO     = 4.0
FILT_HI     = 40.0
FILT_ORD    = 5

# Training
BATCH       = 32
LR          = 0.001
MAX_EP      = 200
PATIENCE    = 30
DROP        = 0.5
SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = ["Left Hand", "Right Hand", "Feet", "Tongue"]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device : {DEVICE}")
print(f"DataDir: {DATA_DIR}")


# ─────────────────────────────────────────────────────────────────────────────
# 1. BANDPASS FILTER
# ─────────────────────────────────────────────────────────────────────────────

def bandpass(X):
    """
    Zero-phase Butterworth 4-40 Hz.
    X: (N, Ch, T)  or  (Ch, T)
    """
    nyq = FS / 2.0
    b, a = sig_proc.butter(FILT_ORD,
                           [FILT_LO / nyq, FILT_HI / nyq],
                           btype='band')
    if X.ndim == 3:
        out = np.empty_like(X)
        for i in range(X.shape[0]):
            for c in range(X.shape[1]):
                out[i, c] = sig_proc.filtfilt(b, a, X[i, c])
        return out
    elif X.ndim == 2:
        out = np.empty_like(X)
        for c in range(X.shape[0]):
            out[c] = sig_proc.filtfilt(b, a, X[c])
        return out
    raise ValueError(f"Unexpected shape {X.shape}")


# ─────────────────────────────────────────────────────────────────────────────
# 2. DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_labels(mat_path):
    mat = sio.loadmat(mat_path)
    return mat["classlabel"].flatten().astype(int) - 1  # 0-indexed


def load_session(gdf_path, mat_path, sid, tag):
    """
    Load one GDF session and segment into trials.
    Returns X (N,22,1125) float32, y (N,) int64
    """
    raw = mne.io.read_raw_gdf(gdf_path, preload=True, verbose=False)

    # Pick 22 EEG channels (drop EOG/Status)
    keep = [ch for ch in raw.ch_names
            if 'EOG' not in ch and 'Status' not in ch
            and 'stim' not in ch.lower()][:N_CH]
    raw.pick_channels(keep)

    events, event_id = mne.events_from_annotations(raw, verbose=False)
    print(f"  [{tag}] events found: {list(event_id.keys())}")

    # Map event codes 769-772 → classes 0-3
    MI = {'769': 0, '770': 1, '771': 2, '772': 3}
    code_map = {}
    for k, v in event_id.items():
        k2 = str(k).strip().replace('.0', '')
        if k2 in MI:
            code_map[v] = MI[k2]
    # Fallback numeric mapping
    if not code_map:
        for v in np.unique(events[:, 2]):
            if v in {7: 0, 8: 1, 9: 2, 10: 3}:
                code_map[v] = {7: 0, 8: 1, 9: 2, 10: 3}[v]
    if not code_map:
        raise ValueError(f"No MI event codes found.\nAvailable: {event_id}")

    data = raw.get_data()  # (22, T_total)
    pre  = int(PRE_S  * FS)       # 62 samples
    post = int(POST_S * FS) + 1   # 1063 → gives exactly 1125 samples

    trials, gdf_lbl = [], []
    for onset, _, code in events:
        if code not in code_map:
            continue
        s, e = onset - pre, onset + post
        if s < 0 or e > data.shape[1]:
            continue
        trials.append(data[:, s:e])
        gdf_lbl.append(code_map[code])

    X = np.array(trials,  dtype=np.float32)
    y_gdf = np.array(gdf_lbl, dtype=np.int64)

    # Prefer .mat labels
    y_mat = load_labels(mat_path)
    if len(y_mat) == len(y_gdf):
        y = y_mat
        src = ".mat"
    else:
        y = y_gdf
        src = "GDF"
    # Ensure exactly TRIAL_SAMP time points — pad or trim if 1 sample off
    if X.shape[2] != TRIAL_SAMP:
        diff = TRIAL_SAMP - X.shape[2]
        if diff == 1:
            X = np.pad(X, ((0,0),(0,0),(0,1)), mode='edge')
        elif diff == -1:
            X = X[:, :, :TRIAL_SAMP]
        print(f"  [{tag}] shape adjusted to {X.shape}")

    print(f"  [{tag}] X={X.shape}  y={y.shape}  labels={src}  "
          f"classes={np.unique(y)}")
    return X, y


def load_subject(sid, data_dir):
    pfx = os.path.join(data_dir, f"A0{sid}")
    Xtr, ytr = load_session(pfx+"T.gdf", pfx+"T.mat", sid, "train")
    Xev, yev = load_session(pfx+"E.gdf", pfx+"E.mat", sid, "eval")
    return Xtr, ytr, Xev, yev


# ─────────────────────────────────────────────────────────────────────────────
# 3. WINDOWING
# ─────────────────────────────────────────────────────────────────────────────

def win_augment(X, y):
    """All 7 windows — training augmentation. (N,22,1125) → (N*7,22,750)"""
    Xo, yo = [], []
    for i in range(len(X)):
        for w in range(N_WIN):
            s = w * STEP_SAMP
            e = s + WIN_SAMP
            if e > X.shape[2]:
                break
            Xo.append(X[i, :, s:e])
            yo.append(y[i])
    Xo = np.array(Xo, dtype=np.float32)
    yo = np.array(yo, dtype=np.int64)
    print(f"  win_augment: {len(X)} trials → {len(Xo)} windows")
    return Xo, yo


def win_center(X, y):
    """Center window only — test/val. (N,22,1125) → (N,22,750)"""
    s = CENTER_WIN * STEP_SAMP   # 3*62 = 186
    e = s + WIN_SAMP             # 186+750 = 936
    Xo = X[:, :, s:e].astype(np.float32)
    print(f"  win_center : {len(X)} trials  window[{s}:{e}]")
    return Xo, y.copy()


# ─────────────────────────────────────────────────────────────────────────────
# 3b. WINDOW VOTING INFERENCE
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def predict_with_voting(model, X_trials, mean, std):
    """
    For each trial extract ALL 7 windows, get softmax scores for each,
    average the scores, then take argmax → final prediction per trial.

    X_trials : (N, 22, 1125) — raw filtered trials (NOT yet windowed)
    Returns  : (N,) predicted class indices
    """
    model.eval()
    N = len(X_trials)
    all_probs = np.zeros((N, N_CLS), dtype=np.float32)

    for w in range(N_WIN):
        s = w * STEP_SAMP
        e = s + WIN_SAMP
        Xw = X_trials[:, :, s:e].astype(np.float32)   # (N, 22, 750)
        Xw = norm(Xw, mean, std)
        xb = torch.FloatTensor(Xw).to(DEVICE)
        logits = model(xb)                              # (N, 4)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs += probs                              # accumulate

    all_probs /= N_WIN                                  # average
    return all_probs.argmax(axis=1)                     # (N,)


# ─────────────────────────────────────────────────────────────────────────────
# 4. NORMALIZATION  (fit on train windows, apply everywhere)
# ─────────────────────────────────────────────────────────────────────────────

def fit_norm(Xtr):
    """Per-channel z-score stats from training windows."""
    mean = Xtr.mean(axis=(0, 2), keepdims=True)   # (1,22,1)
    std  = Xtr.std(axis=(0, 2),  keepdims=True) + 1e-8
    return mean, std


def norm(X, mean, std):
    return (X - mean) / std


# ─────────────────────────────────────────────────────────────────────────────
# 5. SWCNET MODEL
# ─────────────────────────────────────────────────────────────────────────────

class SWCNet(nn.Module):
    """
    Exact architecture from Table III of the paper.
    Input : (B, Ch, Ts)  →  (B, 1, Ch, Ts) internally
    Output: (B, K) logits
    """
    def __init__(self, n_ch=22, n_cls=4, n_t=750, drop=0.5):
        super().__init__()

        # ── Block I: Spatial then Temporal conv ──────────────────────────
        self.sp_conv  = nn.Conv2d(1,  16, kernel_size=(n_ch, 1), bias=False)
        self.bn1      = nn.BatchNorm2d(16)

        # padding=(0,32) keeps temporal dimension unchanged after (1,64) kernel
        self.tm_conv  = nn.Conv2d(16, 32, kernel_size=(1, 64),
                                  padding=(0, 32), bias=False)
        self.bn2      = nn.BatchNorm2d(32)
        self.act1     = nn.ReLU()
        self.pool1    = nn.AvgPool2d((1, 2))

        # ── Block II: Pointwise then two poolings ─────────────────────────
        self.pw_conv  = nn.Conv2d(32, 16, kernel_size=(1, 1), bias=False)
        self.bn3      = nn.BatchNorm2d(16)
        self.act2     = nn.ReLU()
        self.pool2    = nn.AvgPool2d((1, 4))
        self.pool3    = nn.AvgPool2d((1, 8))
        self.dropout  = nn.Dropout(p=drop)

        # Dynamically compute flatten size
        with torch.no_grad():
            flat = self._fwd(torch.zeros(1, 1, n_ch, n_t)).numel()
        self.fc = nn.Linear(flat, n_cls)

    def _fwd(self, x):
        x = self.bn1(self.sp_conv(x))
        x = self.act1(self.bn2(self.tm_conv(x)))
        x = self.pool1(x)
        x = self.act2(self.bn3(self.pw_conv(x)))
        x = self.pool2(x)
        x = self.pool3(x)
        x = self.dropout(x)
        return x

    def forward(self, x):           # x: (B, Ch, T)
        x = x.unsqueeze(1)          # → (B, 1, Ch, T)
        x = self._fwd(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


# ─────────────────────────────────────────────────────────────────────────────
# 6. TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

class EarlyStopping:
    def __init__(self, patience=PATIENCE):
        self.patience  = patience
        self.counter   = 0
        self.best      = np.inf
        self.stop      = False
        self.state     = None

    def step(self, val_loss, model):
        if val_loss < self.best - 1e-5:
            self.best  = val_loss
            self.state = {k: v.clone() for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore(self, model):
        if self.state:
            model.load_state_dict(self.state)


def loader(X, y, shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.LongTensor(y))
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=0)


def train_ep(model, dl, opt, crit):
    model.train()
    ls, cor, tot = 0., 0, 0
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        out = model(xb)
        l   = crit(out, yb)
        l.backward()
        opt.step()
        ls  += l.item() * len(xb)
        cor += (out.argmax(1) == yb).sum().item()
        tot += len(xb)
    return ls / tot, cor / tot


@torch.no_grad()
def eval_ep(model, dl, crit):
    model.eval()
    ls, cor, tot = 0., 0, 0
    preds, trues = [], []
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out = model(xb)
        l   = crit(out, yb)
        ls  += l.item() * len(xb)
        p    = out.argmax(1)
        cor += (p == yb).sum().item()
        tot += len(xb)
        preds.extend(p.cpu().numpy())
        trues.extend(yb.cpu().numpy())
    acc = cor / tot
    f1  = f1_score(trues, preds, average='macro', zero_division=0)
    return ls / tot, acc, f1, np.array(preds), np.array(trues)


def train_model(Xtr, ytr, Xva, yva, tag=""):
    tr_dl = loader(Xtr, ytr, shuffle=True)
    va_dl = loader(Xva, yva, shuffle=False)

    model = SWCNet(N_CH, N_CLS, WIN_SAMP, DROP).to(DEVICE)
    crit  = nn.CrossEntropyLoss()
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='min', factor=0.5, patience=10)
    es = EarlyStopping(PATIENCE)

    history = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}

    for ep in range(1, MAX_EP + 1):
        tr_l, tr_a = train_ep(model, tr_dl, opt, crit)
        va_l, va_a, va_f1, _, _ = eval_ep(model, va_dl, crit)

        sched.step(va_l)
        es.step(va_l, model)

        history["tr_loss"].append(tr_l)
        history["va_loss"].append(va_l)
        history["tr_acc"].append(tr_a)
        history["va_acc"].append(va_a)

        if ep % 10 == 0 or ep == 1:
            lr_now = opt.param_groups[0]['lr']
            print(f"  {tag} Ep{ep:3d} | "
                  f"Tr {tr_l:.4f}/{tr_a*100:.1f}% "
                  f"Va {va_l:.4f}/{va_a*100:.1f}% "
                  f"LR {lr_now:.5f}")

        if es.stop:
            print(f"  {tag} Early stop @ ep{ep} (best={es.best:.4f})")
            break

    es.restore(model)
    return model, history


# ─────────────────────────────────────────────────────────────────────────────
# 7. PLOTTING
# ─────────────────────────────────────────────────────────────────────────────

def plot_cm(y_true, y_pred, title, fname):
    cm  = confusion_matrix(y_true, y_pred, normalize='true',
                           labels=list(range(N_CLS))) * 100
    acc = accuracy_score(y_true, y_pred) * 100
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, vmin=0, vmax=100)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True",      fontsize=11)
    ax.set_title(f"{title}\nAcc={acc:.2f}%  F1={f1:.4f}", fontsize=11)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {fname}")


def plot_hist(hist, title, fname):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    axs[0].plot(hist["tr_loss"], label="Train")
    axs[0].plot(hist["va_loss"], label="Val")
    axs[0].set_title("Loss"); axs[0].set_xlabel("Epoch")
    axs[0].legend(); axs[0].grid(True, alpha=0.3)

    axs[1].plot([a*100 for a in hist["tr_acc"]], label="Train")
    axs[1].plot([a*100 for a in hist["va_acc"]], label="Val")
    axs[1].set_title("Accuracy (%)"); axs[1].set_xlabel("Epoch")
    axs[1].legend(); axs[1].grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# 8. EXPERIMENT SETUP I  (50:30:20, all sessions combined)
# ─────────────────────────────────────────────────────────────────────────────

def setup_I(sid, data_dir):
    print(f"\n{'='*65}")
    print(f"  Subject {sid}  |  Setup I  (50:30:20)")
    print(f"{'='*65}")

    Xtr_raw, ytr, Xev_raw, yev = load_subject(sid, data_dir)

    # Combine both sessions
    X_all = np.concatenate([Xtr_raw, Xev_raw])
    y_all = np.concatenate([ytr,     yev])
    N     = len(X_all)
    print(f"  Total trials: {N}")

    # ── Step 1: Filter ALL data first ────────────────────────────────────
    print("  Bandpass filtering (4-40 Hz)...")
    X_all_f = bandpass(X_all)

    # ── Step 2: Augment ALL data into windows FIRST ───────────────────────
    # Paper splits at window level (50:30:20 of total windows)
    # This is what produces the high ~97% accuracy in the paper
    X_aug, y_aug = win_augment(X_all_f, y_all)
    N_win = len(X_aug)
    print(f"  Total windows after augmentation: {N_win}")

    # ── Step 3: Shuffle and split windows 50:30:20 ───────────────────────
    rng = np.random.RandomState(SEED + sid)
    idx = rng.permutation(N_win)
    X_aug, y_aug = X_aug[idx], y_aug[idx]

    n1 = int(0.50 * N_win)
    n2 = int(0.30 * N_win)
    Xtr_w, ytr_w = X_aug[:n1],      y_aug[:n1]
    Xte_w, yte_w = X_aug[n1:n1+n2], y_aug[n1:n1+n2]
    Xva_w, yva_w = X_aug[n1+n2:],   y_aug[n1+n2:]
    print(f"  Windows — Tr:{len(Xtr_w)} Te:{len(Xte_w)} Va:{len(Xva_w)}")

    # ── Step 4: Normalize (fit on train windows only) ─────────────────────
    mn, sd   = fit_norm(Xtr_w)
    Xtr_n    = norm(Xtr_w, mn, sd)
    Xte_n    = norm(Xte_w, mn, sd)
    Xva_n    = norm(Xva_w, mn, sd)

    print(f"  Shapes  Tr:{Xtr_n.shape} Te:{Xte_n.shape} Va:{Xva_n.shape}")

    # ── Step 5: Train ─────────────────────────────────────────────────────
    model, hist = train_model(Xtr_n, ytr_w, Xva_n, yva_w, tag="[I]")

    # ── Step 6: Evaluate on test windows directly ─────────────────────────
    # No voting needed — test set is already at window level
    crit = nn.CrossEntropyLoss()
    _, te_acc, te_f1, yp, yt = eval_ep(
        model, loader(Xte_n, yte_w, False), crit
    )

    print(f"\n  ✓ Setup I  Acc={te_acc*100:.2f}%  F1={te_f1:.4f}")
    print(f"    Paper S1 target: ~98.64%")

    plot_cm(yt, yp, f"Subject {sid} — Setup I",
            os.path.join(RESULTS_DIR, f"cm_s{sid}_I.png"))
    plot_hist(hist, f"Subject {sid} — Setup I",
              os.path.join(RESULTS_DIR, f"hist_s{sid}_I.png"))

    print("\n" + classification_report(yt, yp,
                                       target_names=CLASS_NAMES, zero_division=0))
    return te_acc, te_f1, yt, yp, hist


# ─────────────────────────────────────────────────────────────────────────────
# 9. EXPERIMENT SETUP II  (10-fold CV on train session, eval session as test)
# ─────────────────────────────────────────────────────────────────────────────

def setup_II(sid, data_dir):
    print(f"\n{'='*65}")
    print(f"  Subject {sid}  |  Setup II  (10-fold CV + held-out eval)")
    print(f"{'='*65}")

    Xtr_raw, ytr, Xev_raw, yev = load_subject(sid, data_dir)

    print("  Bandpass filtering (4-40 Hz)...")
    Xtr_f = bandpass(Xtr_raw)
    Xev_f = bandpass(Xev_raw)

    # Eval session: center window (used as test for ALL folds)
    Xev_w, yev_w = win_center(Xev_f, yev)

    kf = KFold(n_splits=10, shuffle=True, random_state=SEED + sid)
    fold_acc, fold_f1 = [], []
    best_va   = -1
    best_mn   = best_sd = None
    best_st   = None

    for fold, (tri, vai) in enumerate(kf.split(Xtr_f)):
        Xf_tr, yf_tr = Xtr_f[tri], ytr[tri]
        Xf_va, yf_va = Xtr_f[vai], ytr[vai]

        Xf_tr_w, yf_tr_w = win_augment(Xf_tr, yf_tr)
        Xf_va_w, yf_va_w = win_center(Xf_va,  yf_va)

        mn, sd  = fit_norm(Xf_tr_w)
        Xf_tr_n = norm(Xf_tr_w, mn, sd)
        Xf_va_n = norm(Xf_va_w, mn, sd)

        m, _ = train_model(Xf_tr_n, yf_tr_w, Xf_va_n, yf_va_w,
                           tag=f"[II f{fold+1:02d}/10]")

        crit = nn.CrossEntropyLoss()
        _, va_acc, va_f1, _, _ = eval_ep(
            m, loader(Xf_va_n, yf_va_w, False), crit
        )
        fold_acc.append(va_acc)
        fold_f1.append(va_f1)
        print(f"  Fold {fold+1:2d} | Va Acc={va_acc*100:.2f}%  F1={va_f1:.4f}")

        if va_acc > best_va:
            best_va = va_acc
            best_mn, best_sd = mn.copy(), sd.copy()
            best_st = {k: v.clone() for k, v in m.state_dict().items()}

    print(f"\n  CV: {np.mean(fold_acc)*100:.2f}% ± {np.std(fold_acc)*100:.2f}%")

    # ── Final test on held-out eval session with window voting ────────────
    bm = SWCNet(N_CH, N_CLS, WIN_SAMP, DROP).to(DEVICE)
    bm.load_state_dict(best_st)

    yp_vote = predict_with_voting(bm, Xev_f, best_mn, best_sd)
    yt      = yev

    te_acc = accuracy_score(yt, yp_vote)
    te_f1  = f1_score(yt, yp_vote, average='macro', zero_division=0)

    print(f"\n  ✓ Setup II  Acc (voted)={te_acc*100:.2f}%  F1={te_f1:.4f}")
    print(f"    Paper S1 target: ~78.16%")

    plot_cm(yt, yp_vote, f"Subject {sid} — Setup II (voted)",
            os.path.join(RESULTS_DIR, f"cm_s{sid}_II.png"))
    print("\n" + classification_report(yt, yp_vote,
                                       target_names=CLASS_NAMES, zero_division=0))
    return te_acc, te_f1, yt, yp_vote

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    if not os.path.isdir(DATA_DIR):
        raise FileNotFoundError(
            f"\nDataset not found: '{DATA_DIR}'\n"
            f"Please update DATA_DIR at the top of this script."
        )

    print("\n" + "="*65)
    print("  SWCNet v2  |  Subject 1 Test")
    print("  Dataset : BCIC-IV-2a")
    print("="*65)

    results_I, results_II = {}, {}

    for sid in range(1, 10):
        acc1, f1_1, _, _, _ = setup_I(sid, DATA_DIR)
        acc2, f1_2, _, _ = setup_II(sid, DATA_DIR)
        results_I[sid] = (acc1, f1_1)
        results_II[sid] = (acc2, f1_2)

    print("\n" + "="*65)
    print("  ALL SUBJECTS — FINAL SUMMARY")
    print("="*65)
    print(f"  {'Subj':<6} | {'Setup I Acc':>11} {'F1':>8} | {'Setup II Acc':>12} {'F1':>8}")
    print("-"*65)

    for sid in range(1, 10):
        a1, f1 = results_I[sid]
        a2, f2 = results_II[sid]
        print(f"  S{sid:<5} | {a1*100:>10.2f}% {f1:>8.4f} | {a2*100:>11.2f}% {f2:>8.4f}")

    a1m = np.mean([v[0] for v in results_I.values()])
    f1m = np.mean([v[1] for v in results_I.values()])
    a2m = np.mean([v[0] for v in results_II.values()])
    f2m = np.mean([v[1] for v in results_II.values()])

    print("-"*65)
    print(f"  {'Avg':<6} | {a1m*100:>10.2f}% {f1m:>8.4f} | {a2m*100:>11.2f}% {f2m:>8.4f}")
    print(f"\n  Paper avg | {'97.42%':>11} {'0.9742':>8} | {'69.58%':>12} {'0.6958':>8}")
    print("="*65)

    print("\n" + "="*65)
    print("  SUBJECT 1 — FINAL SUMMARY")
    print("="*65)
    print(f"  Setup I   | Acc: {acc1*100:.2f}%  F1: {f1_1:.4f}   (paper ~98.64%)")
    print(f"  Setup II  | Acc: {acc2*100:.2f}%  F1: {f1_2:.4f}   (paper ~78.16%)")
    print(f"\n  All plots saved to: {RESULTS_DIR}/")
    print("="*65)

Device : cuda
DataDir: /content/drive/MyDrive/BCICIV_2a_gdf

  SWCNet v2  |  Subject 1 Test
  Dataset : BCIC-IV-2a

  Subject 1  |  Setup I  (50:30:20)


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


  [train] events found: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]
  [train] X=(288, 22, 1125)  y=(288,)  labels=.mat  classes=[0 1 2 3]


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


  [eval] events found: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('783')]
  [eval] X=(288, 22, 1125)  y=(288,)  labels=.mat  classes=[0 1 2 3]
  Total trials: 576
  Bandpass filtering (4-40 Hz)...
  win_augment: 576 trials → 4032 windows
  Total windows after augmentation: 4032
  Windows — Tr:2016 Te:1209 Va:807
  Shapes  Tr:(2016, 22, 750) Te:(1209, 22, 750) Va:(807, 22, 750)
  [I] Ep  1 | Tr 1.1600/44.7% Va 0.9491/53.0% LR 0.00100
  [I] Ep 10 | Tr 0.4153/83.1% Va 0.3850/85.5% LR 0.00100
  [I] Ep 20 | Tr 0.2346/91.4% Va 0.3686/84.8% LR 0.00100
  [I] Ep 30 | Tr 0.1491/95.3% Va 0.1639/93.8% LR 0.00100
  [I] Ep 40 | Tr 0.0904/97.4% Va 0.0849/97.3% LR 0.00100
  [I] Ep 50 | Tr 0.0777/97.3% Va 0.0927/97.0% LR 0.00100
  [I] Ep 60 | Tr 0.0886/97.5% Va 0.0918/96.9% LR 0.00100
  [I] Ep 70 | Tr 0.0530/98.2% Va 0.0619/98.1% LR 0.00100
  [I] Ep 80 | Tr 0.0336/99.0% Va 0.0359/98.6% LR 0.00050
  [I] Ep 90 | Tr 0.0407/98.7% Va 0.0378/9

...

  [eval] events found: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('783')]
  [eval] X=(288, 22, 1125)  y=(288,)  labels=.mat  classes=[0 1 2 3]
  Total trials: 576
  Bandpass filtering (4-40 Hz)...
  win_augment: 576 trials → 4032 windows
  Total windows after augmentation: 4032
  Windows — Tr:2016 Te:1209 Va:807
  Shapes  Tr:(2016, 22, 750) Te:(1209, 22, 750) Va:(807, 22, 750)
  [I] Ep  1 | Tr 1.2932/36.8% Va 1.0906/53.2% LR 0.00100
  [I] Ep 10 | Tr 0.4703/83.2% Va 0.3726/87.0% LR 0.00100
  [I] Ep 20 | Tr 0.2680/91.1% Va 0.2390/91.2% LR 0.00100
  [I] Ep 30 | Tr 0.2144/92.3% Va 0.1817/93.3% LR 0.00100
  [I] Ep 40 | Tr 0.1620/94.6% Va 0.1518/94.4% LR 0.00100
  [I] Ep 50 | Tr 0.1300/95.7% Va 0.0881/97.5% LR 0.00100
  [I] Ep 60 | Tr 0.1290/95.7% Va 0.0896/97.4% LR 0.00100
  [I] Ep 70 | Tr 0.0806/97.7% Va 0.0664/97.6% LR 0.00050
  [I] Ep 80 | Tr 0.0693/97.6% Va 0.0692/97.6% LR 0.00050
  [I] Ep 90 | Tr 0.0585/98.1% Va 0.0544/9

/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


  [train] events found: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]
  [train] X=(288, 22, 1125)  y=(288,)  labels=.mat  classes=[0 1 2 3]


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


  [eval] events found: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('783')]
  [eval] X=(288, 22, 1125)  y=(288,)  labels=.mat  classes=[0 1 2 3]
  Bandpass filtering (4-40 Hz)...
  win_center : 288 trials  window[186:936]
  win_augment: 259 trials → 1813 windows
  win_center : 29 trials  window[186:936]
  [II f01/10] Ep  1 | Tr 1.3225/34.6% Va 1.3640/31.0% LR 0.00100
  [II f01/10] Ep 10 | Tr 0.3533/87.6% Va 0.4884/79.3% LR 0.00100
  [II f01/10] Ep 20 | Tr 0.1860/93.9% Va 0.4410/86.2% LR 0.00100
  [II f01/10] Ep 30 | Tr 0.1213/95.9% Va 0.6337/82.8% LR 0.00100
  [II f01/10] Ep 40 | Tr 0.0654/98.2% Va 0.5235/79.3% LR 0.00050
  [II f01/10] Ep 50 | Tr 0.0508/98.4% Va 0.5330/86.2% LR 0.00025
  [II f01/10] Early stop @ ep53 (best=0.4223)
  Fold  1 | Va Acc=86.21%  F1=0.8249
  win_augment: 259 trials → 1813 windows
  win_center : 29 trials  window[186:936]
  [II f02/10] Ep  1 | Tr 1.3284/36.7% Va 1.3776/24.1% LR 0.00100
  [II f02/